In [1]:
import os
import sys
from pathlib import Path

# Проверяем, запускался ли уже этот блок
if "IS_ROOT_SET" not in globals():
    project_root = Path.cwd().parent.parent
    os.chdir(project_root)

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    # Создаем флаг-индикатор
    IS_ROOT_SET = True
    print("Корневая директория инициализирована.")
else:
    print("Инициализация уже была выполнена ранее.")

Корневая директория инициализирована.


In [2]:
import time
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO

In [3]:
from mmengine.config import Config
from mmengine.runner import Runner

In [4]:
output_dir = Path("artifacts/metrics")
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "metrics_comparison.csv"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def evaluate_yolo(model_path, test_loader_or_images):
    model = YOLO(model_path)
    
    start_time = time.time()
    results = model.predict(source=test_loader_or_images, device=device, verbose=False)
    end_time = time.time()
    
    total_time = end_time - start_time
    total_frames = len(results)
    fps = total_frames / total_time if total_time > 0 else 0
    
    metrics = model.val(split='test', verbose=False)
    map_score = metrics.box.map 
    map_50_score = metrics.box.map50
    
    return map_score, map_50_score, fps

def evaluate_fcos(config_path, checkpoint_path):
    cfg = Config.fromfile(config_path)
    
    cfg.load_from = checkpoint_path
    
    if 'visualizer' in cfg and 'vis_backends' in cfg.visualizer:
        cfg.visualizer.vis_backends = [dict(type='LocalVisBackend')]
    
    runner = Runner.from_cfg(cfg)
    
    start_time = time.time()
    metrics = runner.test()
    end_time = time.time()
    total_frames = len(runner.test_dataloader.dataset)
    total_time = end_time - start_time
    fps = total_frames / total_time if total_time > 0 else 0
    
    
    map_score = metrics.get('coco/bbox_mAP', 0.0)
    map_50_score = metrics.get('coco/bbox_mAP_50', 0.0)
    
    return map_score, map_50_score, fps

In [6]:
fcos_config = "src/artifacts/fcos_minecraft_50e/fcos_minecraft.py" 

yolo_checkpoint = "src/artifacts/checkpoints/yolo/experiment1/weights/best.pt"
fcos_checkpoint = "src/artifacts/checkpoints/fcos_minecraft_50e/best_coco_bbox_mAP_epoch_33.pth" 

yolo_map, yolo_map50, yolo_fps = evaluate_yolo(yolo_checkpoint, "datasets/yolo_minecraft/images/test")
fcos_map, fcos_map50, fcos_fps = evaluate_fcos(fcos_config, fcos_checkpoint) 

data = {
    "Model": ["FCOS", "YOLO"],
    "mAP": [fcos_map, yolo_map],
    "mAP_50": [fcos_map50, yolo_map50],
    "FPS": [fcos_fps, yolo_fps]
}

df_metrics = pd.DataFrame(data)
df_metrics.to_csv(csv_path, index=False)

print("\nСравнение стандартных метрик моделей:")
display(df_metrics)

Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.1.2+cu121 CUDA:0 (Tesla T4, 14931MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1669.8±1174.3 MB/s, size: 57.9 KB)
val: Scanning /home/ubuntu/minecraft-object-detection/datasets/yolo_minecraft/labels/test... 155 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 155/155 1.1Kit/s 0.1s<0.2s
val: New cache created: /home/ubuntu/minecraft-object-detection/datasets/yolo_minecraft/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.1it/s 2.4s.2ss
                   all        155        351      0.835      0.663      0.759       0.49
Speed: 0.6ms preprocess, 7.9ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /home/ubuntu/minecraft-object-detection/runs/detect/val


/home/ubuntu/minecraft-object-detection/.venv/lib/python3.11/site-packages/torch/utils/cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


06/12 17:16:53 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.15 (main, Jun 11 2026, 04:03:40) [Clang 22.1.3 ]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1020775379
    GPU 0: Tesla T4
    CUDA_HOME: /usr
    NVCC: Cuda compilation tools, release 12.2, V12.2.140
    GCC: cc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.2+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=comp

/home/ubuntu/minecraft-object-detection/.venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/ubuntu/minecraft-object-detection/.venv/lib/python3.11/site-packages/mmengine/visualization/visualizer.py:760: UserWarning: Warning: The bbox is out of bounds, the drawn bbox may not be in the image
  warnings.warn(
/home/ubuntu/minecraft-object-detection/.venv/lib/python3.11/site-packages/mmengine/visualization/visualizer.py:831: UserWarning: Warning: The polygon is out of bounds, the drawn polygon may not be in the image
  warnings.warn(
/home/ubuntu/minecraft-object-detection/.venv/lib/python3.11/site-packages/mmengine/visualization/visualizer.py:508: UserWarning: Warning: The text is out of bounds, the drawn text may not be in the imag

06/12 17:17:30 - mmengine - INFO - Evaluating bbox...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.41s).
Accumulating evaluation results...
DONE (t=0.18s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.284
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.655
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.231
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.103
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.288
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.435
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=300 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.389
 Average

,Model,mAP,mAP_50,FPS
0,FCOS,0.284000,0.655000,4.638367
1,YOLO,0.490454,0.759077,40.128772


In [12]:
import sys
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

def auto_register_cyrillic():
    # Список возможных путей к шрифту Arial в разных ОС
    paths_to_try = [
        # Windows
        "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/arialbd.ttf",
        # Ubuntu / Linux
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        # macOS
        "/Library/Fonts/Arial.ttf",
        "/Library/Fonts/Arial Bold.ttf",
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf"
    ]
    
    regular_font = None
    bold_font = None
    
    # Ищем регулярный шрифт
    for path in paths_to_try:
        if os.path.exists(path) and "bd" not in path.lower() and "bold" not in path.lower():
            regular_font = path
            break
            
    # Ищем жирный шрифт
    for path in paths_to_try:
        if os.path.exists(path) and ("bd" in path.lower() or "bold" in path.lower()):
            bold_font = path
            break

    # Если в системе нет Arial (например, чистый Linux), используем встроенный Helvetica (но он без кириллицы!)
    # Поэтому ставим fallback на стандартные свободные шрифты Linux
    if not regular_font:
        # Для Google Colab или чистых Linux-серверов устанавливаем шрифт одной командой
        os.system("apt-get install -y fonts-liberation > /dev/null 2>&1")
        regular_font = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
        bold_font = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"

    try:
        pdfmetrics.registerFont(TTFont('Arial', regular_font))
        pdfmetrics.registerFont(TTFont('Arial-Bold', bold_font))
        print(f"Успешно подключены шрифты:\nReg: {regular_font}\nBold: {bold_font}")
    except Exception as e:
        print(f"Не удалось зарегистрировать шрифты автоматически: {e}")

# Запускаем автоматическую регистрацию
auto_register_cyrillic()


Успешно подключены шрифты:
Reg: /usr/share/fonts/truetype/dejavu/DejaVuSans.ttf
Bold: /usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf


In [14]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# Пути к файлам
csv_path = Path("artifacts/metrics/metrics_comparison.csv")
df_metrics = pd.read_csv(csv_path)

report_dir = Path("artifacts")
report_dir.mkdir(parents=True, exist_ok=True)
pdf_path = report_dir / "report.pdf"
chart_path = report_dir / "metrics_comparison_charts.png"

# Построение графиков
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df_metrics.plot(x="Model", y=["mAP", "mAP_50"], kind="bar", ax=axes[0], color=["#4f81bd", "#c0504d"])
axes[0].set_title("Сравнение качества (mAP / mAP_50)")
axes[0].set_ylabel("Значение")
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

df_metrics.plot(x="Model", y="FPS", kind="bar", ax=axes[1], color="#9bbb59", legend=False)
axes[1].set_title("Сравнение скорости (FPS)")
axes[1].set_ylabel("Кадры в секунду")
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(chart_path, dpi=300)
plt.close()

# Инициализация PDF-документа
doc = SimpleDocTemplate(str(pdf_path), pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
story = []

styles = getSampleStyleSheet()

# Настройка стилей с поддержкой кириллицы (fontName='Arial' / 'Arial-Bold')
title_style = ParagraphStyle('Title', parent=styles['Heading1'], fontName='Arial-Bold', fontSize=22, leading=26, spaceAfter=15, textColor=colors.HexColor('#1f497d'))
h2_style = ParagraphStyle('H2', parent=styles['Heading2'], fontName='Arial-Bold', fontSize=14, leading=18, spaceBefore=12, spaceAfter=8, textColor=colors.HexColor('#1f497d'))
body_style = ParagraphStyle('Body', parent=styles['Normal'], fontName='Arial', fontSize=10.5, leading=15, spaceAfter=8)

# Формирование контента документа
story.append(Paragraph("Отчёт по проекту: Сравнение YOLO и FCOS в Minecraft", title_style))
story.append(Spacer(1, 5))

story.append(Paragraph("1. Сводные метрики моделей", h2_style))
table_data = [df_metrics.columns.tolist()]
for row in df_metrics.values.tolist():
    table_data.append([row[0], f"{row[1]:.4f}", f"{row[2]:.4f}", f"{row[3]:.2f}"])

t = Table(table_data, colWidths=[100, 100, 100, 100])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1f497d')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('BOTTOMPADDING', (0,0), (-1,0), 6),
    ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#f9f9f9')),
    ('GRID', (0,0), (-1,-1), 1, colors.HexColor('#d9d9d9')),
    ('FONTNAME', (0,0), (-1,-1), 'Arial'), # Шрифт текста внутри таблицы
    ('FONTSIZE', (0,0), (-1,-1), 10),
]))
story.append(t)
story.append(Spacer(1, 10))

story.append(Paragraph("2. Визуальный анализ результатов", h2_style))
story.append(Image(str(chart_path), width=500, height=210))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Анализ метрик качества и гипотезы", h2_style))
quality_text = "Модель <b>YOLO</b> показала явное превосходство по точности локализации (mAP: 0.4905, mAP_50: 0.7591). " \
               "FCOS отстал по общей метрике mAP (0.2840), однако продемонстрировал жизнеспособный результат на mAP_50 (0.6550).<br/><br/>" \
               "<b>Важное замечание:</b> Существует высокая вероятность того, что модель <b>FCOS не дообучилась</b>. " \
               "Anchor-free архитектуры в MMDetection требуют значительно большего количества эпох обучения (от 50-100 и более) " \
               "и агрессивных графиков изменения скорости обучения (Learning Rate), чтобы сойтись на специфичных и «шумных» текстурах Minecraft. " \
               "YOLO же использует сильные встроенные аугментации (Mosaic, MixUp), позволяющие достигать высоких результатов за короткие сессии обучения."
story.append(Paragraph(quality_text, body_style))

story.append(Paragraph("4. Анализ метрик скорости (Инференс)", h2_style))
speed_text = f"Показатели производительности демонстрируют критический разрыв в скорости работы сетей на тестовом стенде. " \
             f"<b>YOLO выдает 40.13 FPS</b>, что делает её полностью применимой в задачах реального времени (интеграция с игровыми ботами, " \
             f"анализ видеотрансляций игрока «на лету»). <b>FCOS выдает 4.64 FPS</b>. Такой низкий фреймрейт обусловлен " \
             f"тяжелым вычислительным бэкэндом MMDetection/MMEngine без дополнительной оптимизации (например, TensorRT/ONNX) " \
             f"и архитектурными особенностями постобработки (NMS) для FCOS."
story.append(Paragraph(speed_text, body_style))

# Сборка готового PDF
doc.build(story)
print(f"Отчет успешно сохранен в {pdf_path}")


Отчет успешно сохранен в artifacts/report.pdf
